In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
counties = pd.read_csv("counties.csv", encoding="latin1")
counties['Real GDP 2023'] = counties['2023']
counties['Real GDP 2022'] = counties['2022']
counties = counties[counties['LineCode'] == 3.0]
counties.rename(columns={"GeoFIPS": "ID", "GeoName": "Name"}, inplace=True)
counties = counties[["ID", "Name", "Region", "Real GDP 2022", "Real GDP 2023"]]
counties['ID'] = counties['ID'].apply(lambda x: int(x.replace('"', '')))
counties = counties[counties['Name'].str.contains(', ')].assign(State = counties['Name'].str.split(', ').str[1])

education = pd.read_csv("Education2023.csv", encoding="latin1")
education = education[education['Attribute'] == "Percent of adults with a bachelor's degree or higher, 2019-23"]
education.rename(columns={"Value": "Percent w/ Bachelor's+"}, inplace=True)
education.rename(columns={"FIPS Code": "ID"}, inplace=True)

education['ID'] = education['ID'].astype(int)

merged_df = counties.merge(
    education[['ID', "Percent w/ Bachelor's+"]], 
    on='ID', 
    how='left'
)

unemployment = pd.read_csv("Unemployment2023.csv", encoding="latin1")
unemployment = unemployment[unemployment['Attribute'] == "Unemployment_rate_2023"]
unemployment.rename(columns={"Value": "Unemployment", "FIPS_Code": "ID"}, inplace=True)

merged_df = merged_df.merge(
    unemployment[['ID', "Unemployment"]], 
    on='ID', 
    how='left'
)


cost = pd.read_excel("fbc_data_2025.xlsx", sheet_name="County")

cost.rename(columns={"Unnamed: 2": "ID", "Unnamed: 20": "Annual Cost", "Unnamed: 4" : "household", "Rankings": "Median Family Income"}, inplace=True)
cost = cost[cost["household"] == "2p1c"]

merged_df = merged_df.merge(
    cost[['ID', "Annual Cost", "Median Family Income"]], 
    on='ID', 
    how='left'
)

merged_df.head()

weather = pd.read_csv("temps.csv", encoding="latin1")

state_fips = {
    "AL": "01", "AK": "02", "AZ": "04", "AR": "05", "CA": "06", "CO": "08",
    "CT": "09", "DE": "10", "FL": "12", "GA": "13", "HI": "15", "ID": "16",
    "IL": "17", "IN": "18", "IA": "19", "KS": "20", "KY": "21", "LA": "22",
    "ME": "23", "MD": "24", "MA": "25", "MI": "26", "MN": "27", "MS": "28",
    "MO": "29", "MT": "30", "NE": "31", "NV": "32", "NH": "33", "NJ": "34",
    "NM": "35", "NY": "36", "NC": "37", "ND": "38", "OH": "39", "OK": "40",
    "OR": "41", "PA": "42", "RI": "44", "SC": "45", "SD": "46", "TN": "47",
    "TX": "48", "UT": "49", "VT": "50", "VA": "51", "WA": "53", "WV": "54",
    "WI": "55", "WY": "56"
}

def convert_to_fips(row):
    state_code, county_code = row.split("-")
    return int(state_fips[state_code] + county_code)

weather['ID'] = weather['ID'].apply(convert_to_fips)

weather.rename(columns={"Value": "Average Temperature"}, inplace=True)

merged_df = merged_df.merge(
    weather[['ID', "Average Temperature"]], 
    on='ID', 
    how='left'
)

jobs = pd.read_csv("employment23.csv", encoding="latin1", dtype={0: str})

# Preserve total county level data only
jobs = jobs[(jobs["own_code"] == 0) & (jobs["agglvl_code"] == 70)]
jobs = jobs[
    ["area_fips", "oty_total_annual_wages_pct_chg", "annual_avg_emplvl", "annual_avg_estabs",
     "avg_annual_pay", "oty_annual_avg_estabs_chg", "oty_annual_avg_estabs_pct_chg",
     "oty_annual_avg_emplvl_chg", "oty_annual_avg_emplvl_pct_chg"]
    ]
jobs.rename(columns={
    "area_fips": "ID", "oty_total_annual_wages_pct_chg": "Pct Wage Change", "annual_avg_emplvl": "Average Employment Level",
    "annual_avg_estabs": "Average Number of Establishments", "avg_annual_pay": "Average Annual Pay",
    "oty_annual_avg_estabs_chg": "Establishment Change", "oty_annual_avg_estabs_pct_chg": "Establishment Change %",
    "oty_annual_avg_emplvl_chg": "Employment Level Change", "oty_annual_avg_emplvl_pct_chg": "Employment Change %"}, inplace=True
    )

jobs = jobs[jobs.iloc[:, 0].str.isdigit()]
jobs.iloc[:, 0] = jobs.iloc[:, 0].astype(int)

merged_df = merged_df.merge(
    jobs[['ID', 'Pct Wage Change', 'Average Employment Level',
       'Average Number of Establishments', 'Average Annual Pay',
        'Establishment Change %', 'Employment Change %']], 
    on='ID', 
    how='left'
)

pop = pd.read_csv("population.csv", encoding="latin1", low_memory=False)

pop = pop[pop["Unit"] == "Number of persons"]
pop = pop[['GeoFIPS','2022', '2023']]
pop['GeoFIPS'] = pop['GeoFIPS'].apply(lambda x: int(x.replace('"', '')))
pop['GeoFIPS'] = pop['GeoFIPS'].astype(int)

pop = pop[(pop['2022'] != '(NA)') & (pop['2023'] != '(NA)')]

pop['2022'] = pop['2022'].astype(int)
pop['2023'] = pop['2023'].astype(int)

pop['Population Change'] = pop['2023'] - pop['2022']
pop['Population Change %'] = (pop['Population Change'] / pop['2022']) * 100
pop.rename(columns={"GeoFIPS": "ID", "2023": "Population"}, inplace=True)

merged_df = merged_df.merge(
    pop[['ID', 'Population', 'Population Change %']], 
    on='ID', 
    how='left'
)

counties_cleaned = merged_df.dropna()

counties_cleaned = counties_cleaned[counties_cleaned['Average Annual Pay'] != 0]

counties_cleaned['Real GDP 2023'] = counties_cleaned['Real GDP 2023'].astype(int)
counties_cleaned['Real GDP 2022'] = counties_cleaned['Real GDP 2022'].astype(int)
counties_cleaned['GDP Growth %'] = ((counties_cleaned['Real GDP 2023'] - counties_cleaned['Real GDP 2022']) / counties_cleaned['Real GDP 2022']) * 100
counties_cleaned.drop(columns=['Real GDP 2023', 'Real GDP 2022'], inplace=True)

region_mapping = {
    '1': 'New_England',
    '2': 'Atlantic',
    '3': 'Great_Lakes',
    '4': 'Midwest',
    '5': 'Southeast',
    '6': 'Southwest',
    '7': 'Mountains',
    '8': 'West_Coast'
}

counties_cleaned['Region'] = counties_cleaned['Region'].replace(region_mapping)

counties_cleaned = pd.get_dummies(counties_cleaned, columns=["Region"], drop_first=True)
counties_cleaned = pd.get_dummies(counties_cleaned, columns=["State"], drop_first=True)

counties_cleaned.head()


,ID,Name,Percent w/ Bachelor's+,Unemployment,Annual Cost,Median Family Income,Average Temperature,Pct Wage Change,Average Employment Level,Average Number of Establishments,...,State_TN,State_TX,State_UT,State_VA,State_VA*,State_VT,State_WA,State_WI,State_WV,State_WY
0,1001,"Autauga, AL",28.282680,2.2,85436.375,83452,65.7,10.1,11871.0,1073.0,...,False,False,False,False,False,False,False,False,False,False
1,1003,"Baldwin, AL",32.797637,2.3,88936.68,92918,68.9,7.5,82764.0,8302.0,...,False,False,False,False,False,False,False,False,False,False
2,1005,"Barbour, AL",11.464715,4.4,72222.984,60474,66.0,0.8,7702.0,591.0,...,False,False,False,False,False,False,False,False,False,False
3,1007,"Bibb, AL",11.468207,2.5,77889.281,67222,64.2,5.3,4815.0,433.0,...,False,False,False,False,False,False,False,False,False,False
4,1009,"Blount, AL",15.579030,2.1,76866.641,76388,62.8,3.4,8857.0,932.0,...,False,False,False,False,False,False,False,False,False,False


In [6]:
# Exlporatory Data Analysis

print(counties_cleaned.shape)
print(counties_cleaned.dtypes)
print(counties_cleaned.head())

numeric_cols = counties_cleaned.select_dtypes(include=['number']).columns.tolist()

print(numeric_cols)

print(counties_cleaned.isna().sum())

(3044, 72)
ID                         object
Name                       object
Percent w/ Bachelor's+    float64
Unemployment              float64
Annual Cost                object
                           ...   
State_VT                     bool
State_WA                     bool
State_WI                     bool
State_WV                     bool
State_WY                     bool
Length: 72, dtype: object
     ID         Name  Percent w/ Bachelor's+  Unemployment Annual Cost  \
0  1001  Autauga, AL               28.282680           2.2   85436.375   
1  1003  Baldwin, AL               32.797637           2.3    88936.68   
2  1005  Barbour, AL               11.464715           4.4   72222.984   
3  1007     Bibb, AL               11.468207           2.5   77889.281   
4  1009   Blount, AL               15.579030           2.1   76866.641   

  Median Family Income  Average Temperature  Pct Wage Change  \
0                83452                 65.7             10.1   
1               

In [ ]:
counties_cleaned.hist(bins=30, figsize=(20, 15))
plt.tight_layout()
plt.show()

In [ ]:
corr_df = counties_cleaned.drop(columns=['ID', 'Name',])

corr_matrix = corr_df.corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

plt.figure(figsize=(20, 18))
sns.heatmap(corr_matrix, mask=mask, cmap='coolwarm', vmax=1.0, vmin=-1.0, center=0,
            square=True, linewidths=.5, annot=False, fmt='.2f')
plt.title('Correlation Matrix')
plt.show()

In [ ]:
corr_matrix = counties_cleaned.corr(numeric_only=True).abs()

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

# Find features with correlation greater than 0.8
high_corr = [(column, row) for column in upper.columns for row in upper.index if (upper.loc[row, column] > 0.8)]

print(high_corr)

In [ ]:
counties_cleaned = counties_cleaned.drop(columns={'Average Number of Establishments', 'Average Employment Level'})

numeric_cols = counties_cleaned.select_dtypes(include=['float64', 'int64']).columns

for col in numeric_cols:
    plt.figure(figsize=(8, 4))
    sns.histplot(counties_cleaned[col], kde=True)
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.show()

In [ ]:
outcome_vars = ['Population Change %', 'Employment Change %', 'Establishment Change %', 'GDP Growth %']

print(counties_cleaned[outcome_vars].corr())

explanatory_vars = [col for col in counties_cleaned.columns if col not in outcome_vars]

corr_matrix = counties_cleaned[explanatory_vars + outcome_vars].corr(numeric_only=True)

def get_top_predictors(outcome_var, corr_matrix, outcome_vars, top_n=10):
    return corr_matrix[outcome_var].drop(
        [col for col in outcome_vars if col != outcome_var and col in corr_matrix.index]
    ).sort_values(key=abs, ascending=False).head(top_n)

# Example usage:
pop_change_corr = get_top_predictors('Population Change %', corr_matrix, outcome_vars)
empl_change_corr = get_top_predictors('Employment Change %', corr_matrix, outcome_vars)
estab_change_corr = get_top_predictors('Establishment Change %', corr_matrix, outcome_vars)
gdp_change_corr = get_top_predictors('GDP Growth %', corr_matrix, outcome_vars)

print("Top predictors for Population Change %:")
print(pop_change_corr.head(10))

print("Top predictors for Employment Change %:")
print(empl_change_corr.head(10))

print("Top predictors for Establishment Change %:")
print(estab_change_corr.head(10))

print("Top predictors for GDP Growth %:")
print(gdp_change_corr.head(10))

In [ ]:
def find_dummy_columns(df):
    return [col for col in df.columns if df[col].dropna().nunique() == 2 and set(df[col].dropna().unique()).issubset({0, 1})]

# Find dummy variables
dummy_vars = find_dummy_columns(counties_cleaned)

# Set of features to remove (outcomes + dummies)
remove_vars = set(outcome_vars + dummy_vars)

# Explanatory variables: all columns minus outcomes and dummies
explanatory_vars = [col for col in counties_cleaned.columns if col not in remove_vars]

corr_matrix = counties_cleaned[explanatory_vars + outcome_vars].corr(numeric_only=True)

# Example usage:
pop_change_corr_nd = get_top_predictors('Population Change %', corr_matrix, outcome_vars)
empl_change_corr_nd = get_top_predictors('Employment Change %', corr_matrix, outcome_vars)
estab_change_corr_nd = get_top_predictors('Establishment Change %', corr_matrix, outcome_vars)
gdp_change_corr_nd = get_top_predictors('GDP Growth %', corr_matrix, outcome_vars)

print("Top predictors for Population Change %:")
print(pop_change_corr_nd)

print("\nTop predictors for Employment Change %:")
print(empl_change_corr_nd)

print("\nTop predictors for Establishment Change %:")
print(estab_change_corr_nd)

print("\nTop predictors for GDP Growth %:")
print(gdp_change_corr_nd)



In [ ]:
output_features = [
    'Population Change %',
    'Employment Change %',
    'Establishment Change %',
    'GDP Growth %'
]

# Set the plotting style
sns.set(style="whitegrid")

# Loop through each feature and plot histogram with skew
for feature in output_features:
    plt.figure(figsize=(8, 5))
    sns.histplot(counties_cleaned[feature], kde=True, bins=30, color='steelblue')
    skew_value = counties_cleaned[feature].skew()
    plt.title(f"{feature} (Skewness = {skew_value:.2f})", fontsize=14)
    plt.xlabel(feature)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

In [ ]:
counties_cleaned.to_csv('counties_v1.csv', index=False)